# 07 — Unsupervised Representation Learning

This notebook rebuilds the unsupervised representation-learning section for the repo's canonical 29-asset universe.

The goal is structural diagnosis: correlation geometry, PCA factors, eigenportfolio-style factor portfolios, clustering, and rolling stability. It does **not** create a new alpha model, allocation engine, SWITCH rule, VMP overlay, supervised target, or out-of-sample performance claim.

## §0 Setup and project context

Unsupervised representation learning sits after the ML, DL, and RL sections as a diagnostic layer. The methods below use only asset returns and existing metadata to understand latent risk structure in the 29-asset panel.

Static full-sample PCA is treated as an ex-post descriptive diagnostic. Rolling PCA uses only the historical data inside each rolling window.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import squareform

SEED = 42
np.random.seed(SEED)
TRADING_DAYS = 252

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent.resolve()

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from aiam.data.universe import UNIVERSE_29
from aiam.features.asset_class import ASSET_CLASS_MAP

OUT_DIR = ROOT / "results" / "notebook_07"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print(f"ROOT: {ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"Output directory: {OUT_DIR.relative_to(ROOT)}")

## §1 Load canonical 29-asset panel

The notebook uses the repo-local canonical cache: daily adjusted prices and daily returns for the 29-asset universe. Published OHLCV data documents provenance, but the first-pass analysis uses the existing parquet cache to avoid web access and keep reruns fast.

Monthly regime signals are loaded only for the optional interpretive overlay in §8.

In [ ]:
prices_path = ROOT / "data" / "cache" / "prices_29.parquet"
returns_path = ROOT / "data" / "cache" / "returns_29_2003_2026.parquet"
# Existing regime artifact from the earlier macro-regime / practitioner workflow.
# The upstream regime engine classifies eight macro/market indicators
# (GDP, VIX, SPX, CPI, unemployment, 10Y, 2Y, and yield-curve slope)
# and derives dominant_regime as the modal regime across indicators.
# Notebook 07 does not rebuild that engine and does not use regimes to
# select strategies or allocations. The labels are loaded only as a descriptive
# overlay for PCA/eigenportfolio diagnostics.
regime_path = ROOT / "data" / "cache" / "regime_signals_2003_2026.parquet"

prices = pd.read_parquet(prices_path)
returns = pd.read_parquet(returns_path)
prices.index = pd.to_datetime(prices.index)
returns.index = pd.to_datetime(returns.index)

prices = prices.reindex(columns=UNIVERSE_29)
returns = returns.reindex(columns=UNIVERSE_29)

assert prices.shape[1] == 29, f"Expected 29 price columns, got {prices.shape[1]}"
assert returns.shape[1] == 29, f"Expected 29 return columns, got {returns.shape[1]}"
assert list(returns.columns) == list(UNIVERSE_29), "Return columns must match UNIVERSE_29 order"

asset_meta = pd.DataFrame({
    "asset": UNIVERSE_29,
    "asset_class": [ASSET_CLASS_MAP.get(asset, "unknown") for asset in UNIVERSE_29],
}).set_index("asset")

regimes = None
if regime_path.exists():
    regimes = pd.read_parquet(regime_path)
    regimes.index = pd.to_datetime(regimes.index)

print(f"Prices:  {prices.shape}, {prices.index.min().date()} to {prices.index.max().date()}")
print(f"Returns: {returns.shape}, {returns.index.min().date()} to {returns.index.max().date()}")
print(f"Assets:  {returns.shape[1]}")
print(f"Regime signals loaded: {regimes is not None}")

asset_inventory = pd.DataFrame(index=returns.columns)
asset_inventory.index.name = "asset"
asset_inventory["asset_class"] = asset_meta["asset_class"]
asset_inventory["first_valid_return"] = returns.apply(lambda s: s.first_valid_index())
asset_inventory["last_valid_return"] = returns.apply(lambda s: s.last_valid_index())
asset_inventory["non_null_returns"] = returns.notna().sum()
asset_inventory["missing_returns"] = returns.isna().sum()
asset_inventory.to_csv(OUT_DIR / "asset_inventory.csv")
asset_inventory

## §2 Return matrix and standardization

For static descriptive PCA and clustering, this pass uses complete daily rows across all 29 assets. That makes the covariance/correlation geometry deterministic and avoids model-specific imputation. The tradeoff is that the common sample starts only after the shorter-history assets are available.

For rolling PCA later, each window is standardized using only observations inside that historical window.

In [ ]:
returns_matrix = returns.dropna(how="all").copy()
complete_returns = returns_matrix.dropna(axis=0, how="any").copy()

means = complete_returns.mean()
stds = complete_returns.std(ddof=1)
zero_vol_assets = stds[stds <= 0].index.tolist()
assert not zero_vol_assets, f"Zero-volatility assets cannot be standardized: {zero_vol_assets}"

standardized_returns = (complete_returns - means) / stds

return_summary = pd.DataFrame({
    "asset_class": asset_meta["asset_class"],
    "mean_daily_return": complete_returns.mean(),
    "annualized_volatility": complete_returns.std(ddof=1) * np.sqrt(TRADING_DAYS),
    "missing_returns_full_panel": returns_matrix.isna().sum(),
    "complete_sample_observations": complete_returns.notna().sum(),
})
return_summary.to_csv(OUT_DIR / "return_summary.csv")

standardization_check = pd.DataFrame({
    "standardized_mean": standardized_returns.mean(),
    "standardized_std": standardized_returns.std(ddof=1),
})

print(f"Full return panel rows after all-null drop: {len(returns_matrix):,}")
print(f"Complete-case static analysis rows: {len(complete_returns):,}")
print(f"Complete-case date range: {complete_returns.index.min().date()} to {complete_returns.index.max().date()}")
print("Standardization check, max abs mean:", round(float(standardization_check["standardized_mean"].abs().max()), 12))
print("Standardization check, std range:", standardization_check["standardized_std"].min().round(6), standardization_check["standardized_std"].max().round(6))
return_summary.head(10)

## §3 Correlation structure and asset-class map

Correlation is the first structural view of the universe. Assets are ordered by the repo's canonical asset-class taxonomy so that within-class and cross-class blocks are visible.

In [ ]:
CLASS_ORDER = [
    "us_single_stock",
    "us_sector_etf",
    "broad_equity_etf",
    "intl_equity_etf",
    "fixed_income_etf",
    "commodity_etf",
    "fx_spot",
]

corr = complete_returns.corr()
corr.to_csv(OUT_DIR / "correlation_matrix.csv")

ordered_assets = (
    asset_meta.assign(class_order=asset_meta["asset_class"].map({c: i for i, c in enumerate(CLASS_ORDER)}))
    .sort_values(["class_order"], kind="stable")
    .index.tolist()
)
ordered_corr = corr.loc[ordered_assets, ordered_assets]

summary_rows = []
for left in CLASS_ORDER:
    left_assets = asset_meta.index[asset_meta["asset_class"] == left].tolist()
    if not left_assets:
        continue
    for right in CLASS_ORDER:
        right_assets = asset_meta.index[asset_meta["asset_class"] == right].tolist()
        if not right_assets:
            continue
        block = corr.loc[left_assets, right_assets].copy()
        if left == right:
            mask = ~np.eye(len(left_assets), dtype=bool)
            vals = block.to_numpy()[mask]
        else:
            vals = block.to_numpy().ravel()
        vals = vals[np.isfinite(vals)]
        summary_rows.append({
            "asset_class_left": left,
            "asset_class_right": right,
            "average_correlation": float(np.mean(vals)) if len(vals) else np.nan,
            "observations": int(len(vals)),
        })
asset_class_corr_summary = pd.DataFrame(summary_rows)
asset_class_corr_summary.to_csv(OUT_DIR / "asset_class_correlation_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(ordered_corr.values, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(ordered_assets)))
ax.set_yticks(range(len(ordered_assets)))
ax.set_xticklabels(ordered_assets, rotation=90, fontsize=8)
ax.set_yticklabels(ordered_assets, fontsize=8)
ax.set_title("Canonical 29-Asset Return Correlation Matrix")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Correlation")
fig.tight_layout()
fig.savefig(FIG_DIR / "correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

asset_class_corr_summary.pivot(index="asset_class_left", columns="asset_class_right", values="average_correlation").round(3)

## §4 PCA on asset returns

PCA is fitted on the standardized complete-case return matrix. The sign of each principal component is arbitrary, so the implementation applies a deterministic sign convention: the asset with the largest absolute loading is made positive. This makes tables and plots stable, but it does not give the positive side intrinsic economic priority.

In [ ]:
def apply_component_sign_convention(components: np.ndarray, scores: np.ndarray | None = None):
    adjusted = components.copy()
    adjusted_scores = None if scores is None else scores.copy()
    signs = []
    anchors = []
    for i in range(adjusted.shape[0]):
        anchor_idx = int(np.argmax(np.abs(adjusted[i])))
        sign = 1.0 if adjusted[i, anchor_idx] >= 0 else -1.0
        adjusted[i] *= sign
        if adjusted_scores is not None:
            adjusted_scores[:, i] *= sign
        signs.append(sign)
        anchors.append(standardized_returns.columns[anchor_idx])
    return adjusted, adjusted_scores, signs, anchors

n_components = len(UNIVERSE_29)
pca = PCA(n_components=n_components, random_state=SEED)
pca_scores_raw = pca.fit_transform(standardized_returns)
components, pca_scores, component_signs, component_anchors = apply_component_sign_convention(
    pca.components_, pca_scores_raw
)

pc_names = [f"PC{i}" for i in range(1, n_components + 1)]
pca_scores = pd.DataFrame(pca_scores, index=standardized_returns.index, columns=pc_names)

explained = pd.DataFrame({
    "component": pc_names,
    "explained_variance": pca.explained_variance_,
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_explained_variance_ratio": np.cumsum(pca.explained_variance_ratio_),
    "sign_anchor_asset": component_anchors,
})
explained.to_csv(OUT_DIR / "pca_explained_variance.csv", index=False)

loadings = pd.DataFrame(components.T, index=standardized_returns.columns, columns=pc_names)
loadings.index.name = "asset"
loadings = loadings.join(asset_meta)
loadings.to_csv(OUT_DIR / "pca_loadings.csv")

fig, ax1 = plt.subplots(figsize=(10, 5))
first_n = 12
ax1.bar(range(1, first_n + 1), explained.loc[: first_n - 1, "explained_variance_ratio"], color="#4C78A8")
ax1.set_xlabel("Principal component")
ax1.set_ylabel("Explained variance ratio")
ax1.set_title("PCA Explained Variance")
ax2 = ax1.twinx()
ax2.plot(range(1, first_n + 1), explained.loc[: first_n - 1, "cumulative_explained_variance_ratio"], marker="o", color="#F58518")
ax2.set_ylabel("Cumulative explained variance")
ax2.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(FIG_DIR / "pca_explained_variance.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 9))
loading_matrix = loadings.loc[ordered_assets, pc_names[:5]].to_numpy()
im = ax.imshow(loading_matrix, cmap="coolwarm", vmin=-np.abs(loading_matrix).max(), vmax=np.abs(loading_matrix).max())
ax.set_xticks(range(5))
ax.set_xticklabels(pc_names[:5])
ax.set_yticks(range(len(ordered_assets)))
ax.set_yticklabels(ordered_assets, fontsize=8)
ax.set_title("PCA Loadings, First Five Components")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Loading")
fig.tight_layout()
fig.savefig(FIG_DIR / "pca_loading_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

explained.head(10).round(4)

In [ ]:
interpretation_rows = []
for pc in pc_names[:3]:
    s = loadings[pc].sort_values()
    interpretation_rows.append({
        "component": pc,
        "explained_variance_ratio": explained.loc[explained["component"] == pc, "explained_variance_ratio"].iloc[0],
        "largest_positive_loadings": ", ".join(s.tail(5).index.tolist()),
        "largest_negative_loadings": ", ".join(s.head(5).index.tolist()),
    })
pc_interpretation = pd.DataFrame(interpretation_rows)
pc_interpretation.to_csv(OUT_DIR / "pca_component_interpretation.csv", index=False)
pc_interpretation

The table above supports economic interpretation from the largest loading assets. Because PCA signs are arbitrary, the positive and negative sides are labels under the sign convention, not an inherent long/short recommendation.

## §5 Eigenportfolio construction and interpretation

Eigenportfolio weights are derived directly from PCA loadings. Each component loading vector is normalized so gross exposure, the sum of absolute weights, equals 1. These are diagnostic factor portfolios used to inspect latent structure. They are not production strategies and are not compared to the strategy leaderboard.

In [ ]:
eigen_names = [f"EigenPC{i}" for i in range(1, 6)]
eigen_weights = pd.DataFrame(index=standardized_returns.columns)
for i, name in enumerate(eigen_names):
    w = pd.Series(components[i], index=standardized_returns.columns, dtype=float)
    w = w / w.abs().sum()
    eigen_weights[name] = w

eigen_weights.index.name = "asset"
eigen_weights_with_meta = eigen_weights.join(asset_meta)
eigen_weights_with_meta.to_csv(OUT_DIR / "eigenportfolio_weights.csv")

eigen_returns = complete_returns @ eigen_weights
eigen_returns.index.name = "date"
eigen_returns.to_parquet(OUT_DIR / "eigenportfolio_returns.parquet")

eigen_summary = pd.DataFrame({
    "mean_daily_return": eigen_returns.mean(),
    "annualized_volatility": eigen_returns.std(ddof=1) * np.sqrt(TRADING_DAYS),
    "diagnostic_annualized_sharpe": eigen_returns.mean() / eigen_returns.std(ddof=1) * np.sqrt(TRADING_DAYS),
    "min_daily_return": eigen_returns.min(),
    "max_daily_return": eigen_returns.max(),
    "gross_exposure": eigen_weights.abs().sum(),
    "net_exposure": eigen_weights.sum(),
})
eigen_summary.to_csv(OUT_DIR / "eigenportfolio_summary.csv")

fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=True)
axes = axes.ravel()
for i, name in enumerate(eigen_names):
    w = eigen_weights[name].sort_values()
    colors = np.where(w.values >= 0, "#4C78A8", "#E45756")
    axes[i].bar(w.index, w.values, color=colors)
    axes[i].set_title(name)
    axes[i].tick_params(axis="x", rotation=90, labelsize=7)
    axes[i].axhline(0, color="black", linewidth=0.8)
axes[-1].axis("off")
fig.suptitle("Eigenportfolio Weights, Gross Exposure Normalized to 1", y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "eigenportfolio_weights.png", dpi=150, bbox_inches="tight")
plt.show()

cum_eigen = (1 + eigen_returns).cumprod()
fig, ax = plt.subplots(figsize=(11, 6))
for col in eigen_names[:5]:
    ax.plot(cum_eigen.index, cum_eigen[col], label=col)
ax.set_title("Eigenportfolio Cumulative Diagnostic Returns")
ax.set_ylabel("Cumulative value of factor return series")
ax.legend(ncol=3)
fig.tight_layout()
fig.savefig(FIG_DIR / "eigenportfolio_cumulative_returns.png", dpi=150, bbox_inches="tight")
plt.show()

eigen_summary.round(4)

## §6 Clustering as diversification map

Assets are clustered using a correlation-distance transform:

\[
D_{ij} = \sqrt{0.5(1 - \rho_{ij})}
\]

This creates peer/diversification groups from return co-movement. Cluster labels are descriptive and are not used as allocation signals.

In [ ]:
distance = np.sqrt(0.5 * (1 - corr.clip(-1, 1)))
distance_matrix = distance.to_numpy(copy=True)
np.fill_diagonal(distance_matrix, 0.0)
condensed_distance = squareform(distance_matrix, checks=False)
link = linkage(condensed_distance, method="average")

n_clusters = 6
cluster_ids = fcluster(link, t=n_clusters, criterion="maxclust")
cluster_labels = pd.DataFrame({
    "asset": corr.index,
    "asset_class": asset_meta.loc[corr.index, "asset_class"].values,
    "cluster": cluster_ids,
}).sort_values(["cluster", "asset_class", "asset"]).reset_index(drop=True)
cluster_labels.to_csv(OUT_DIR / "cluster_labels.csv", index=False)

cluster_summary = (
    cluster_labels.groupby("cluster")
    .agg(n_assets=("asset", "count"), assets=("asset", lambda x: ", ".join(x)), asset_classes=("asset_class", lambda x: ", ".join(sorted(set(x)))))
    .reset_index()
)
cluster_summary.to_csv(OUT_DIR / "cluster_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 6))
dendro = dendrogram(link, labels=corr.index.tolist(), leaf_rotation=90, leaf_font_size=8, ax=ax, color_threshold=None)
ax.set_title("Hierarchical Clustering Dendrogram, Correlation Distance")
ax.set_ylabel("Distance")
fig.tight_layout()
fig.savefig(FIG_DIR / "cluster_dendrogram.png", dpi=150, bbox_inches="tight")
plt.show()

cluster_ordered_assets = dendro["ivl"]
clustered_corr = corr.loc[cluster_ordered_assets, cluster_ordered_assets]
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(clustered_corr.values, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(cluster_ordered_assets)))
ax.set_yticks(range(len(cluster_ordered_assets)))
ax.set_xticklabels(cluster_ordered_assets, rotation=90, fontsize=8)
ax.set_yticklabels(cluster_ordered_assets, fontsize=8)
ax.set_title("Cluster-Ordered Correlation Matrix")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Correlation")
fig.tight_layout()
fig.savefig(FIG_DIR / "clustered_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

cluster_summary

## §7 Rolling PCA stability diagnostics

Rolling PCA checks whether the latent structure is stable through time. Each 756-trading-day window is standardized and fitted independently using only returns inside that historical window. PC1 loading stability is measured as the absolute cosine similarity to the previous window's PC1, which handles arbitrary PCA sign flips.

In [ ]:
window = 756
step = 21
rolling_rows = []
prev_pc1 = None

for end_pos in range(window, len(complete_returns) + 1, step):
    window_returns = complete_returns.iloc[end_pos - window:end_pos]
    window_std = window_returns.std(ddof=1)
    if window_std.le(0).any():
        continue
    x = (window_returns - window_returns.mean()) / window_std
    p = PCA(n_components=3, random_state=SEED)
    p.fit(x)
    comps = p.components_.copy()
    for i in range(comps.shape[0]):
        anchor_idx = int(np.argmax(np.abs(comps[i])))
        if comps[i, anchor_idx] < 0:
            comps[i] *= -1
    pc1 = comps[0]
    if prev_pc1 is None:
        pc1_similarity = np.nan
    else:
        denom = np.linalg.norm(pc1) * np.linalg.norm(prev_pc1)
        pc1_similarity = float(abs(np.dot(pc1, prev_pc1) / denom)) if denom else np.nan
    rolling_rows.append({
        "window_end": window_returns.index[-1],
        "window_start": window_returns.index[0],
        "observations": len(window_returns),
        "pc1_explained_variance_ratio": p.explained_variance_ratio_[0],
        "pc2_explained_variance_ratio": p.explained_variance_ratio_[1],
        "pc3_explained_variance_ratio": p.explained_variance_ratio_[2],
        "pc1_pc3_cumulative_explained_variance_ratio": p.explained_variance_ratio_[:3].sum(),
        "pc1_abs_cosine_similarity_to_previous_window": pc1_similarity,
    })
    prev_pc1 = pc1

rolling_pca = pd.DataFrame(rolling_rows)
rolling_pca.to_csv(OUT_DIR / "rolling_pca_diagnostics.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 6))
for col, label in [
    ("pc1_explained_variance_ratio", "PC1"),
    ("pc2_explained_variance_ratio", "PC2"),
    ("pc3_explained_variance_ratio", "PC3"),
]:
    ax.plot(pd.to_datetime(rolling_pca["window_end"]), rolling_pca[col], label=label)
ax.set_title("Rolling PCA Explained Variance, 756-Day Historical Windows")
ax.set_ylabel("Explained variance ratio")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "rolling_pca_explained_variance.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(pd.to_datetime(rolling_pca["window_end"]), rolling_pca["pc1_abs_cosine_similarity_to_previous_window"], color="#4C78A8")
ax.set_title("Rolling PC1 Loading Stability")
ax.set_ylabel("Absolute cosine similarity to previous window")
ax.set_ylim(0, 1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "rolling_pc1_stability.png", dpi=150, bbox_inches="tight")
plt.show()

rolling_pca_preview = rolling_pca.tail().copy()
numeric_cols = rolling_pca_preview.select_dtypes(include="number").columns
rolling_pca_preview[numeric_cols] = rolling_pca_preview[numeric_cols].round(4)
rolling_pca_preview

## §8 Optional: PCA factors by existing macro regime

The `dominant_regime` labels used in this section are inherited from the repo's earlier macro-regime and practitioner analytics workflow. They are generated outside this notebook by the regime engine in `src/aiam/data/regimes/regime_engine.py` and the extended 2003-2026 build script `scripts/build_regime_signals_2003.py`, then cached in `data/cache/regime_signals_2003_2026.parquet`. The same published-style structure is also available in `data/published/regime_signals.parquet`.

The regime engine builds eight indicator-level regimes from GDP, VIX, SPX, CPI, unemployment, the 10-year Treasury yield, the 2-year Treasury yield, and yield-curve slope; `dominant_regime` is then derived as the modal regime across those indicators. Earlier notebooks use these labels for regime-aware diagnostics and SWITCH/VMP strategy comparisons, especially `notebooks/02_practitioner_analytics.ipynb` and the SWITCH artifact `data/cache/portfolio_returns/switch_v2a_oos_29assets.parquet`.

Notebook 07 does not reconstruct that regime engine and does not introduce a new regime-switching strategy. Here, regimes are used only as an interpretive overlay for PCA/eigenportfolio diagnostics. The labels are aligned to daily returns by reindexing monthly observations onto daily dates and forward-filling after the monthly observation date.

In [ ]:
if regimes is None or "dominant_regime" not in regimes.columns:
    print("Regime file unavailable or missing dominant_regime; skipping §8 artifacts.")
    regime_summary = pd.DataFrame()
else:
    daily_regime = regimes["dominant_regime"].dropna().sort_index()
    daily_regime = daily_regime.reindex(eigen_returns.index, method="ffill")
    regime_joined = eigen_returns.join(daily_regime.rename("dominant_regime")).dropna(subset=["dominant_regime"])
    regime_joined["dominant_regime"] = regime_joined["dominant_regime"].astype(int)

    summary = []
    for regime, group in regime_joined.groupby("dominant_regime"):
        for col in eigen_names[:3]:
            r = group[col].dropna()
            summary.append({
                "dominant_regime": int(regime),
                "factor": col,
                "count": int(r.count()),
                "mean_daily_return": r.mean(),
                "annualized_volatility": r.std(ddof=1) * np.sqrt(TRADING_DAYS),
                "diagnostic_annualized_sharpe": r.mean() / r.std(ddof=1) * np.sqrt(TRADING_DAYS) if r.std(ddof=1) > 0 else np.nan,
            })
    regime_summary = pd.DataFrame(summary)
    regime_summary.to_csv(OUT_DIR / "pca_by_regime_summary.csv", index=False)

    plot_data = regime_summary.pivot(index="dominant_regime", columns="factor", values="mean_daily_return") * TRADING_DAYS
    fig, ax = plt.subplots(figsize=(10, 6))
    plot_data.plot(kind="bar", ax=ax)
    ax.set_title("Eigenportfolio Mean Return by Existing Dominant Regime")
    ax.set_ylabel("Annualized mean, diagnostic only")
    ax.set_xlabel("Existing dominant_regime")
    ax.legend(title="Factor")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "pca_by_regime.png", dpi=150, bbox_inches="tight")
    plt.show()

regime_summary.head(12).round(4)

## §9 Research interpretation and limitations

Notebook 07 adds a structural representation layer for the canonical 29-asset universe. Correlations show the raw geometry of co-movement, PCA compresses that geometry into orthogonal linear factors, eigenportfolio-style weights expose the asset composition of those factors, clustering maps peer/diversification groups, and rolling PCA tests whether these representations are stable through time.

Important limitations:

- PCA is sample-dependent; leading components can change as the estimation window changes.
- PCA signs are arbitrary. This notebook uses a deterministic sign convention for readability, not economic directionality.
- Clusters are unstable across windows, linkage rules, and distance definitions.
- Correlation regimes change, especially around crises and rate-shock periods.
- Eigenportfolios are diagnostic factor portfolios, not automatically tradable production strategies.
- Unsupervised learning is diagnostic, not alpha proof.
- The macro-regime overlay is descriptive only and is not a strategy-selection engine.

In [ ]:
created_files = sorted(p.relative_to(ROOT).as_posix() for p in OUT_DIR.rglob("*") if p.is_file())
print("Notebook 07 artifacts:")
for path in created_files:
    print(f"- {path}")